# R2-CAP · Capstone — modelo predictivo sobre datos de tu organismo — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea B · *Data Scientist* · Semana 17–18 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Plantear un **problema de ML** público y su uso.
2. Elegir **features sin fuga de datos**.
3. Entrenar y **evaluar honestamente** un modelo.
4. Documentarlo en una **model card**.

**Competencia de salida:** ejecutar el ciclo de vida de un modelo de forma responsable, de datos a model card.

**Dato real:** compras públicas (ejemplo) o el dataset de tu organismo.

**Entregable:** que las **3 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "compras_ml.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/compras_ml.csv", CSV)

df = pd.read_csv(CSV)
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

## 1. Enmarca el problema de modelado

Define en una frase la **decisión pública** que el modelo va a apoyar y por qué un modelo aporta
(¿priorizar fiscalización? ¿anticipar demanda?). Ejemplo guía: *predecir si una orden la adjudicará
un proveedor grande, para entender la concentración del mercado.*

> Usamos `compras_ml.csv` como ejemplo; reemplázalo por los datos de tu organismo.
> **Objetivo (ejemplo):** clasificar si el proveedor es **Grande**.

## 2. Features sin fuga de datos

El error más caro: incluir una variable que **es** la respuesta. Aquí `tamano_num` codifica el tamaño del proveedor (el target) → **queda fuera**. Selecciona features honestas.

### ✍️ Ejercicio 1 — Define X, y y la partición — sin fuga

In [ ]:
FEATURES = ["cantidad", "monto_total", "categoria", "region_comprador"]   # NO tamano_num (fuga)
y = (df["tamano_proveedor"] == "Grande").astype(int)
X = df[FEATURES]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print("train:", Xtr.shape, "| positivos:", round(100 * y.mean(), 1), "%")

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert "tamano_num" not in FEATURES
    assert "tamano_proveedor" not in FEATURES
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Usa cantidad, monto_total, categoria y region_comprador; nunca tamano_num/tamano_proveedor.')
    print("   Detalle:", e)

## 3. Entrena el modelo

Un pipeline con one-hot de las categóricas + un clasificador. El pipeline evita fugas en el preprocesamiento.

### ✍️ Ejercicio 2 — Entrena un pipeline

In [ ]:
pre = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), ["categoria", "region_comprador"])],
    remainder="passthrough")
modelo = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))]).fit(Xtr, ytr)
print("modelo entrenado:", hasattr(modelo, "predict"))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert hasattr(modelo, "predict")
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))]).fit(Xtr, ytr).")
    print("   Detalle:", e)

## 4. Evalúa honestamente

Mide en el conjunto de **prueba** con una métrica adecuada al desbalance (F1).

### ✍️ Ejercicio 3 — Calcula el F1 en test

In [ ]:
pred = modelo.predict(Xte)
f1 = f1_score(yte, pred, zero_division=0)
print("F1 (test):", round(f1, 3))
print(classification_report(yte, pred, zero_division=0))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert 0.0 <= f1 <= 1.0
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'f1_score(yte, pred, zero_division=0).')
    print("   Detalle:", e)

## 5. Model card (ficha del modelo)

Completa esta ficha — acompaña al modelo cuando lo entregas:

> **Nombre / versión:** _(p.ej. clasificador-proveedor-grande v1)_
> **Objetivo y uso previsto:** _(qué decide, quién lo usa)_
> **Datos de entrenamiento:** compras públicas (ChileCompra); features: cantidad, monto, categoría, región.
> **Métrica y desempeño:** F1 = _(valor)_ en test; clase positiva ≈ _(%)_.
> **Limitaciones y sesgos:** _(grupos donde rinde peor; qué NO predice)_
> **Riesgos / uso responsable:** _(no usar para decisiones punitivas automáticas; revisión humana)_

## Rúbrica de evaluación

| Criterio | Qué se evalúa |
|---|---|
| **Framing** | El problema y el uso del modelo están bien planteados. |
| **Calidad del dato** | Limpieza y features razonables. |
| **Evaluación honesta** | Sin **fuga de datos**; métrica adecuada al desbalance; mide en test. |
| **Model card** | Documenta datos, desempeño, límites y uso responsable. |
| **Reproducibilidad** | El notebook corre de principio a fin. |

> **Entregable:** notebook reproducible + modelo entrenado + **model card** + informe de validación.